# SVC Processing Pipeline — Interactive Tutorial

This notebook runs the **whole pipeline** on SVC HR-1024i `.sig` scans and draws a
plot at every step. It assumes that you already have a folder of raw `.sig` files.
You only need to tell the settings cell where that folder is.

| Part | What it does |
|---|---|
| 1 — Single spectrum | Load → inspect raw → process → visualize each step |
| 2 — Full folder | Load → filter references & outliers → process → save |
| 3 — Pairs (by position) | Average consecutive scans → plot individuals + means |
| 4 — Groups (by scan number) | Average by the scan number in each filename (real-data way) |

Every code cell is explained in the text or comments just above it. Run the cells
top to bottom with **Shift + Enter**.

## Instrument overview

The SVC HR-1024i is a field spectroradiometer that records reflectance across the
visible, near-infrared, and shortwave-infrared regions using **three detector
arrays**:

| Detector array | Region | Approximate range |
|---|---:|---:|
| Silicon | VNIR | 340–1012 nm |
| InGaAs | SWIR-1 | 972–1910 nm |
| Extended InGaAs | SWIR-2 | 1894–2517 nm |

Because the arrays overlap, each raw `.sig` file contains three sequential sensor
segments with small discontinuities that must be trimmed, matched, smoothed, and
resampled into one continuous curve from 400–2500 nm.

## 1. Setup

The first code cell checks whether the notebook helpers are already available. If not,
it installs `svc-processing` and the plotting dependencies into the current kernel. No
repository clone or manual Python-path setup is required.

You can download this notebook by itself, or clone the repository to get the tutorial,
documentation, configs, and tests together:

```bash
git clone https://github.com/regs08/SVCProcessingPipeline.git
cd SVCProcessingPipeline
jupyter lab notebooks/pipeline_demo.ipynb
```

The public repository intentionally does **not** include raw field `.sig` files because
instrument headers can contain GPS and time metadata. Copy your authorized scans to a
folder you control. If you place them under a cloned repo's ignored `data/` directory,
you can locate candidate folders from a terminal with:

```bash
find data -type f -name '*.sig' | head
```

Use the containing folder—not an individual file—as `DATA_FOLDER` below.

In [ ]:
# Install the pipeline and plotting support into this notebook's Python kernel.
# This notebook is meant to run even when downloaded by itself.
import importlib
import subprocess
import sys

PYPI_SPEC = "svc-processing[demo]>=0.1.5"
SOURCE_SPEC = (
    "svc-processing[demo] @ "
    "https://github.com/regs08/SVCProcessingPipeline/archive/refs/heads/main.zip"
)


def _notebook_api_available():
    try:
        importlib.import_module("pipeline.notebook")
    except ModuleNotFoundError:
        return False
    return True


def _pip_install(spec):
    print(f"Installing {spec}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", spec])


if not _notebook_api_available():
    try:
        _pip_install(PYPI_SPEC)
    except subprocess.CalledProcessError:
        print(
            "The released package with notebook helpers is not available from PyPI yet. "
            "Installing from the public GitHub source archive instead."
        )
        _pip_install(SOURCE_SPEC)

importlib.invalidate_caches()
if not _notebook_api_available():
    raise RuntimeError(
        "The setup cell finished, but pipeline.notebook is still unavailable. "
        "Restart the kernel, run this setup cell again, and confirm that the public "
        "package/source you are installing contains svc-processing 0.1.5 or newer."
    )

print("svc-processing notebook helpers are available.")

In [ ]:
import csv
import matplotlib.pyplot as plt
import os
from pathlib import Path

try:
    from pipeline.processor import GroupSpec
    from pipeline.notebook import (
        build_config,            # turn a few settings into a pipeline config
        Spectrum,                # one scan
        SpectraCollection,       # a folder of scans
        save_spectra_csv,        # write processed spectra to CSV
        average_pairs,           # Part 3: average scans by position
        plot_paired_averages,
        average_groups,          # Part 4: average scans by scan number
        plot_groups,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import pipeline.notebook. Run the setup/install cell above in this "
        "same kernel, then rerun this import cell."
    ) from exc

print("Helpers imported from the installed svc-processing package.")

### Your settings — edit these

Set `DATA_FOLDER` to the folder containing your raw `.sig` files. Absolute paths are
simplest; examples are `Path("/Users/name/data/run_01")` on macOS/Linux and
`Path(r"C:\Users\name\data\run_01")` on Windows. A relative path such as
`Path("data/run_01")` is resolved from the folder where Jupyter is running.

Change `OUTPUT_FOLDER` if you want results somewhere else. Leave
`INSTRUMENT = "auto"` to read the instrument from the file headers, or set it
explicitly to `"bronze"` or `"silver"`.

In [ ]:
WORK_FOLDER = Path.cwd()

# REQUIRED: replace the fallback path with the folder containing your raw .sig files.
# SVC_DATA_FOLDER is optional and is mainly useful for automated/headless runs.
DATA_FOLDER = Path(
    os.environ.get("SVC_DATA_FOLDER", "/Users/nr466/Python Projects/svcProcessingPipeline/notebooks/pipeline_demo/demo")
).expanduser()

# Results are written beneath the folder where this notebook is running.
OUTPUT_FOLDER = WORK_FOLDER / "pipeline_outputs/notebook_run"

# "auto" detects the instrument from the file headers; or use "bronze" / "silver".
INSTRUMENT = "auto"

### Stage 1 end line (where each scan gets trimmed)

`END_LINE` is the wavelength where **Stage 1** cuts off each `.sig` file. Determine it
after instrument calibration or when introducing a new SVC instrument: below `data=`,
find the maximum value in the **first (wavelength) column**, check that value across
several files, and preserve the exact decimal string.

Leave `END_LINE = None` when the installed calibrated default matches your instrument
(**2520.4 nm** for bronze, **2517.9 nm** for silver). Otherwise enter the maximum as a
string, such as `END_LINE = "2517.9"`.

In [ ]:
# Stage 1 truncation wavelength.
#   None      -> use the instrument's calibrated default (bronze 2520.4, silver 2517.9)
#   "2517.9"  -> trim every scan at this wavelength instead
# A custom value must exactly match a wavelength in the first data column; the nearest
# value is not used. A mismatch leaves trailing rows untrimmed and may cause a later
# warning about finding more than the HR-1024i's three sensor sweeps.
END_LINE = None

### Check the input folder

Before building the configuration, confirm that `DATA_FOLDER` exists and contains
`.sig` files directly inside it. Keep scans from different instruments in separate
folders.

In [ ]:
sig_files = sorted(DATA_FOLDER.glob("*.sig"))
if not sig_files:
    raise FileNotFoundError(
        f"No .sig files found in {DATA_FOLDER}. Edit DATA_FOLDER in the settings cell "
        "to point at the folder containing your scans."
    )
print(f"Found {len(sig_files)} .sig files in {DATA_FOLDER}")
print("First file:", sig_files[0].name)

### Build the config

`build_config()` bundles those settings, detects which instrument took the scans, and
fills in the parity-verified Stage 2 parameters. Printing `config` shows what it
resolved.

In [ ]:
config = build_config(
    data_folder=DATA_FOLDER,
    output_folder=OUTPUT_FOLDER,
    instrument=INSTRUMENT,
    end_line=END_LINE,
)
config            # show a friendly summary: instrument, end line, paths, processing params

### Prepare the files — Stage 1

`config.prepare()` truncates each raw `.sig` file at the end wavelength shown in the
config summary above (`END_LINE`, or the instrument default) and writes the result into
the processed folder. This is **Stage 1** of the pipeline; every step after this reads
the truncated files.

If `prepare()` warns that the end line wasn't found in your files, set `END_LINE` in the
settings cell and re-run from there.

In [ ]:
config.prepare()

---
## Part 1 — a single spectrum, step by step

Loading one scan makes each stage easy to see.

The summary below doubles as an **instrument check**:
- `sensor count = 3` confirms a three-array instrument (Si + InGaAs + extended InGaAs)
- the `splice wavelengths` near ~984 nm and ~1896 nm are the array boundaries

If those look unexpected, double-check `DATA_FOLDER` before processing everything.

In [ ]:
# Load the first Stage 1 output and print a summary.
processed_scans = sorted(config.processed_folder.glob("*.sig"))
spectrum = Spectrum.from_config(config, processed_scans[0])
print(spectrum)

### Raw

Plotted in file order, the wavelength axis **folds back twice** — once near 1000 nm
and once near 1890 nm — because each new detector array starts at a lower wavelength
than the previous one ended. That fold-back is the artifact the pipeline corrects.

In [ ]:
spectrum.plot()        # raw reflectance, in file order

### Process — Stage 2

`process()` runs Stage 2: trim the sensor overlaps, align the detectors with a
multiplicative splice correction, Gaussian-smooth, and resample onto the clean
400–2500 nm grid (using the parameters from your config).

In [ ]:
spectrum.process()
print(spectrum)        # now "processed : True", output on the 400-2500 nm grid

### Processing steps

Three panels show the spectrum at each stage. Red dashed lines mark the splice
wavelengths.

In [ ]:
spectrum.plot_processing_steps()

---
## Part 2 — a whole folder

The same pipeline now runs on every scan. Two filters first remove scans that should
not be analysed:
- **Reference panels** — the white Spectralon target (reflectance ≈ 1.0 everywhere).
- **Outliers** — scans whose mean reflectance is far from the group (e.g. obstructed
  view, instrument not settled).

In [ ]:
# Load every processed scan; the collection honours the config's Stage 2 settings.
collection = SpectraCollection.from_config(config)
print(collection)

### Raw — all scans

The raw plot shows the sensor fold-backs for every scan. Reference panels appear as
nearly flat lines up near reflectance 1.0.

In [ ]:
collection.plot_raw()

### Filter

Remove the reference panels first, then the outliers (so the panels don't skew the
outlier statistics). Each call prints how many scans it removed.

In [ ]:
collection.filter_reference_scans()    # drop white-reference panels
collection.filter_outliers()           # drop scans far from the group mean

### Process every scan

In [ ]:
collection.process()                   # Stage 2 on every remaining scan
print(collection)

### Check one scan

A three-panel before/after on one scan confirms the pipeline ran correctly.

In [ ]:
collection.plot_processing_steps(spectrum_index=0)

### All cleaned spectra

In [ ]:
collection.plot()                      # every cleaned scan overlaid

### Save

Write the cleaned spectra to CSV: one row per scan, wavelength columns 400–2500 nm.
We keep the path in `spectra_csv` for the grouping steps below.

In [ ]:
spectra_csv = save_spectra_csv(collection, config.output_folder / "spectra.csv")
print("Saved:", spectra_csv)

---
## Part 3 — averaging repeat scans (by position)

Field work often takes several scans per sample. The simplest way to average them is
by **position** in the list — e.g. scans 0 and 1 are one sample, 2 and 3 the next.
Quick and intuitive; see Part 4 for the robust, real-data approach.

### Which position is which file?

In [ ]:
# Position (index) -> filename, so you can choose groups deliberately.
for i, s in enumerate(collection.spectra):
    print(f"[{i}] {s.name}")

### Define pairs and average

Each tuple contains **0-based positions** in the filtered collection. The example
below makes consecutive pairs dynamically, so it remains valid if the number of scans
changes. Edit the tuples when your measurement design uses different groupings.

In [ ]:
# Pair consecutive positions: (0, 1), (2, 3), ...
groups = [
    tuple(range(i, min(i + 2, len(collection.spectra))))
    for i in range(0, len(collection.spectra), 2)
]

print("Groups to average:", groups)
pairs = average_pairs(collection, groups=groups)
print(pairs)

Individual scans are drawn faded behind their bold group mean — one colour per
pair — so you can see both within-pair agreement and between-pair variation.

In [ ]:
plot_paired_averages(collection, pairs, groups=groups)

### Save the pairs

In [ ]:
# One row per pair, wavelength columns 400-2500 nm.
paired_csv = config.output_folder / "spectra_paired.csv"
pairs.to_csv(paired_csv)
print("Saved:", paired_csv)

---
## Part 4 — grouping by scan number (real data)

Real datasets don't group by position — they group by the **scan number** baked into
each filename (the trailing number, e.g. `...0003` → scan 3). Grouping by number is
robust to filtering and reordering, and it's exactly how the production pipeline and
the `naming_ids/` lookup tables work.

### Define leaf groups by scan number

The default demo groups come from `notebooks/pipeline_demo/mapping.csv`, which maps
each scan number to a condition and a `leaf_number`. We group by leaf number within
each condition, so the two scans for each leaf are averaged before comparing
`symptomatic` and `asymptomatic` leaves.

In [ ]:
# Scan number = the trailing number in each filename.
surviving_scan_numbers = {int(s.name.split(".")[-1]) for s in collection.spectra}

mapping_csv = Path("notebooks/pipeline_demo/mapping.csv")
if not mapping_csv.exists():
    mapping_csv = Path("pipeline_demo/mapping.csv")
if not mapping_csv.exists():
    raise FileNotFoundError("Could not find notebooks/pipeline_demo/mapping.csv")

# mapping.csv has one row per scan, a condition label, and a leaf_number.
leaf_groups = {}
with mapping_csv.open(newline="") as handle:
    reader = csv.DictReader(handle, skipinitialspace=True)
    field_lookup = {field.strip().lower(): field for field in (reader.fieldnames or [])}
    missing_fields = {"scan_no", "symptomatic", "leaf_number"} - set(field_lookup)
    if missing_fields:
        raise KeyError(f"mapping.csv is missing required column(s): {sorted(missing_fields)}")

    scan_field = field_lookup["scan_no"]
    condition_field = field_lookup["symptomatic"]
    leaf_field = field_lookup["leaf_number"]

    for row in reader:
        scan_number = int(row[scan_field])
        condition = row[condition_field].strip().lower()
        leaf_number = int(row[leaf_field])
        if scan_number in surviving_scan_numbers:
            leaf_groups.setdefault((condition, leaf_number), []).append(scan_number)

condition_order = ["asymptomatic", "symptomatic"]
number_groups = []
for condition in condition_order:
    leaf_numbers = sorted(
        leaf_number
        for group_condition, leaf_number in leaf_groups
        if group_condition == condition
    )
    for leaf_number in leaf_numbers:
        scans = tuple(sorted(leaf_groups[(condition, leaf_number)]))
        number_groups.append(
            GroupSpec(members=scans, name=f"{condition} leaf {leaf_number}")
        )

for group in number_groups:
    print(f"{group.name}: {group.members}")

`average_groups()` reads the saved CSV and averages the two scans for each leaf.
The plot below draws one line per averaged leaf, coloured by condition, plus a bold
mean line for each condition.

In [ ]:
grouped = average_groups(spectra_csv, number_groups)

# Plot only the leaf averages so the two conditions are easy to compare.
wavelength_columns = [col for col in grouped.columns if str(col).isdigit()]
wavelengths = [int(col) for col in wavelength_columns]
condition_colors = {"asymptomatic": "steelblue", "symptomatic": "crimson"}

fig, ax = plt.subplots(figsize=(12, 5))
for _, row in grouped.iterrows():
    group_name = str(row["name"])
    condition = group_name.split(" leaf ")[0]
    values = row[wavelength_columns].astype(float).to_numpy()
    ax.plot(
        wavelengths,
        values,
        color=condition_colors.get(condition, "gray"),
        alpha=0.45,
        lw=1.4,
    )

for condition in condition_order:
    condition_rows = grouped[
        grouped["name"].astype(str).str.startswith(f"{condition} leaf ")
    ]
    if condition_rows.empty:
        continue
    condition_mean = condition_rows[wavelength_columns].astype(float).mean(axis=0)
    ax.plot(
        wavelengths,
        condition_mean.to_numpy(),
        color=condition_colors.get(condition, "gray"),
        lw=3.0,
        label=f"{condition} mean ({len(condition_rows)} leaves)",
    )

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.set_title("Leaf averages by condition")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Save the groups

In [ ]:
grouped_csv = config.output_folder / "spectra_grouped.csv"
grouped.to_csv(grouped_csv, index=False)
print("Saved:", grouped_csv)

---
## (Optional) Save your settings as a config file

`config.to_json()` writes a `config/config.json`-compatible file. You can hand the
same file to the command-line pipeline for batch processing:
`svc-pipeline path/to/my_config.json`.

In [ ]:
config_path = config.to_json(config.output_folder / "my_config.json")
print("Wrote:", config_path)